# 02 - Baseline System Evaluation
**Establishing the Pretrained Performance Floor**

Before fine-tuning, we must evaluate the un-tuned baseline (`BAAI/bge-base-en-v1.5`) at its native 768 dimensions against the SciFact test set to quantify future improvements.


In [45]:
import os
import sys

# 1. Escape back to /content root
%cd /content

# 2. Clone repo if not already present
if not os.path.exists("/content/matryoshka-domain-rag"):
    !git clone https://github.com/premsaipusapati-debug/matryoshka-domain-rag.git

# 3. Copy src and configs to /content so they are always accessible
!cp -rf /content/matryoshka-domain-rag/src /content/
!cp -rf /content/matryoshka-domain-rag/configs /content/

# 4. Add /content to Python path
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# 5. Imports
import yaml
import pandas as pd
from src.data_loader import load_scifact_raw, get_eval_data
from src.model import load_embedding_model
from src.evaluate import evaluate_retrieval

# 6. Load config
with open("/content/configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("🎉 SUCCESS! All modules and config loaded cleanly!")


/content
🎉 SUCCESS! All modules and config loaded cleanly!


In [46]:
import sys
import os
import yaml
import pandas as pd

# 1. Copy src and configs directly into Colab's root /content directory
!cp -rf /content/matryoshka-domain-rag/src /content/
!cp -rf /content/matryoshka-domain-rag/configs /content/

# 2. Ensure /content is at the very front of Python's search path
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# 3. Now import from src
from src.data_loader import load_scifact_raw, get_eval_data
from src.model import load_embedding_model
from src.evaluate import evaluate_retrieval

# 4. Load config.yaml directly from /content/configs/
with open("/content/configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("🎉 SUCCESS! All modules and config loaded cleanly!")


🎉 SUCCESS! All modules and config loaded cleanly!


In [47]:
# Use 'mteb/scifact' as the dataset name to avoid deprecated HF loading scripts
raw_data = load_scifact_raw("mteb/scifact")
corpus_dict, queries_dict, qrels_dict = get_eval_data(raw_data, split="test")

print(f"Evaluation Test Set:")
print(f"- Documents in Corpus: {len(corpus_dict)}")
print(f"- Test Queries: {len(queries_dict)}")

Evaluation Test Set:
- Documents in Corpus: 5183
- Test Queries: 300


In [48]:
base_model_name = config["model"]["base_model"]
print(f"Loading un-tuned baseline model: {base_model_name}")

baseline_model = load_embedding_model(
    model_name_or_path=base_model_name,
    max_seq_length=config["model"]["max_seq_length"]
)

Loading un-tuned baseline model: BAAI/bge-base-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [49]:
import os
import sys
import pandas as pd

# Forcefully clean sys.path to avoid stale references
sys.path = [p for p in sys.path if "matryoshka" not in p and p != "/content"]

# Insert absolute paths securely at the front
sys.path.insert(0, "/content/matryoshka-domain-rag")
sys.path.insert(0, "/content")

# Flush sys.modules cache for 'src' to force python to look up the directory again
for mod in list(sys.modules.keys()):
    if mod == "src" or mod.startswith("src."):
        sys.modules.pop(mod, None)

from src.model import load_embedding_model
from src.evaluate import evaluate_retrieval
from src.data_loader import load_scifact_raw, get_eval_data

# Ensure config dictionary exists in the environment
if 'config' not in globals():
    config = {
        "dataset": {
            "name": "scifact"
        },
        "model": {
            "base_model": "BAAI/bge-base-en-v1.5",
            "max_seq_length": 512
        },
        "evaluation": {
            "top_k": [1, 3, 5, 10, 100]
        }
    }

# Ensure dataset is loaded in the active workspace
if 'corpus_dict' not in globals() or 'queries_dict' not in globals() or 'qrels_dict' not in globals():
    print("Dataset not found in global memory. Loading 'mteb/scifact' dataset dynamically...")
    raw_data = load_scifact_raw("mteb/scifact")
    corpus_dict, queries_dict, qrels_dict = get_eval_data(raw_data, split="test")

# Ensure baseline_model exists in the environment
if 'baseline_model' not in globals():
    base_model_name = config["model"]["base_model"]
    print(f"Loading un-tuned baseline model dynamically: {base_model_name}")
    baseline_model = load_embedding_model(
        model_name_or_path=base_model_name,
        max_seq_length=config["model"]["max_seq_length"]
    )

# Monkeypatch the typo in the library where it calls model.ncode instead of model.encode
if not hasattr(baseline_model, "ncode"):
    baseline_model.ncode = baseline_model.encode

baseline_metrics = evaluate_retrieval(
    corpus_dict=corpus_dict,
    queries_dict=queries_dict,
    qrels_dict=qrels_dict,
    model=baseline_model,
    target_dim=768,
    top_k=config["evaluation"]["top_k"]
)

baseline_df = pd.DataFrame([baseline_metrics])
baseline_df.insert(0, "Model", "Pretrained BGE-base (768d)")

print("\n--- Baseline Performance Floor ---")
display(baseline_df)

# Save baseline metrics
output_dir = "/content/matryoshka-domain-rag/output" if os.path.exists("/content/matryoshka-domain-rag") else "output"
os.makedirs(output_dir, exist_ok=True)
out_path = os.path.join(output_dir, "baseline_metrics.csv")
baseline_df.to_csv(out_path, index=False)
print(f"Saved baseline metrics to: {out_path}")

Batches:   0%|          | 0/81 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]


--- Baseline Performance Floor ---


,Model,Recall@10,MRR@10,nDCG@10,Avg_Latency_ms,Storage_per_1M_MB,Dimension
0,Pretrained BGE-base (768d),0.8767,0.7004,0.7376,0.09,2929.7,768


Saved baseline metrics to: /content/matryoshka-domain-rag/output/baseline_metrics.csv
